# Дообучение языковой модели (Supervised Fine-tuning)
Модель, обученную в режиме самообучения на некотором универсальном датасете, будем назвывать предобученной (pretrained). Такая модель хорошо умеет предсказывать следующий токен в предложении, обладает богатыми «общими знаниями» о языке и мире, но совершенно не пригодна для выполнения узких специфических задач: она не умеет следовать инструкциям, не знает доменную специфику, не умеет форматировать ответы в требуемом формате и тоне.

Для того чтобы побороть это ограничение, её нужно дообучить под конкретную задачу. Благо дообучение сильно дешевле, чем обучение с нуля. Здесь можно провести аналогию с учебой: научившись читать и писать, и овладев базовыми навыками, изучить новый предмет становится кратно легче. Собрать дом из готового кирпича легче, чем обжигать каждый кирпич в отдельности

Как донастроить модель под нужную нам задачу:
- Zero-shot learning<br>мы никак не модифицируем модель, надеемся что она "из коробки" сумеет сгенерировать устраивающий нас ответ; если задача не сложная или качество ответа не критичный показатель, это бывает вполне оправданно<br><br>
- Few-shot learning<br>мы добавляем в промпт модели один или несколько примеров в том формате, который от нее ожидаем ("сделай вот как здесь", "отоформатируй ответ вот так и так"). Интеллекта модели хватает, чтобы неплохо обобщить требования из нескольких примеров<br><br>
- Supervised Fine-tuning<br>мы проводим дополнительный цикл обучения модели на относительно небольшом размеченном наборе данных (1-100K примеров). Идея в том, что модель уже обладает фундаментальными языковыми навыками, и нам остаётся лишь донастроить её под задачу

Дообучать модель можно как под одну задачу, так и сразу под несколько (Multi-task Fine-tuning)

Несмотря на меньшие требования по кол-ву примеров по сравнению с предобучением в self-supervised резиме, это все же обучение с учителем и данные нужно размечать руками, что делает процоесс крайне дорогим:
- дорого по памяти<br>цикл обучения требует хранить веса, градиенты, состояния оптимизатора. Даже для небольших моделей это 10-100GB видеопамяти<br><br>
- дорого по занимаемому месту на диске<br>каждая дообученная версия — это полная копия исходной модели. Все версии нужно где-то хранить, даже если там поменялась пара весов<br><br>
- есть проблема забывания (Catastrophic forgetting)<br>при долгом обучении модель может «забывать» часть исходных способностей. Мы же хотим чтобы модель "накапливала" знаяния, а не переписывала их

Феномен *Catastrophic forgetting* впервые описан [(MacCloskey et al, 1989)](https://www.andywills.info/hbab/mccloskeycohen.pdf) на примере сетей, обучавшихся сложению двух чисел - когда после обучения складывать однозначные числа модели обучали сложению двузначных, они полностью разучивались складывать однозначные. Иными словами, посмотрел рилс - забыл дату куликовской битвы. В реальности конечно чуть сложнее, но тем не менее - обучение тем более если оно многошаговое, гетерогеннок и растянутое по времени, нужно грамотно планировать.

В контексте обучения LLM моделей проблема больше актуальна для этапа Preference Optimization, но для SFT тоже [добавить иллюстрацию]. Авторы прорывной модели ChatGPT InstructGPT из OpenAI называли этот феномен *alignment tax* - подстраивание модели под пользователя происходит всегда за счет некоторой потери точности

Полезность этапа дообучения с учителем многократно ислледовалась. Напмриер, в работе [(Shi et al., 2023)](https://arxiv.org/abs/2308.04014) посчитали влияние дообучения на качество модели LLaMA‑7B. На целевом датасете оно ожидаемо выросло, а вот общие навыки по бенчмарку MMLU упали на –3.5 процентных пункта

Некоторые методы борьбы с забыванием:
- *Replay / Mixing*<Br>При дообучении на новом датасете периодически подмешивают данные из pre‑training датасета<br><Br>
- *Регуляризация*<br>вариант, предложенный [(Kirkpatrick et al, 2017)](https://arxiv.org/pdf/1612.00796) который назвали Elastic Weight Consolidation (EWC) - в функцию потерь добавляется штраф за большое изменение параметра (было vs стало), оцененный через Fisher Information <br>другой вариант использовали OpenAI при разработке InstructGPT в 2022 году, там добавили в качестве штрафа расстояние Кульбака-Либлера между распределениями выходных токенов<br><br>
- *LoRA и адаптеры*<Br>обучают только небольшие низкоранговые матрицы, не трогая основные веса. Значительно снижает забывание, но при агрессивном обучении всё же возможно; подробнее о методах репараметризации ниже<br><Br>
- *Мультзадачное обучение* (multi-task learning)<Br>если дообучать модель нужно сразу под несколько задач, лучше делать это не последовательно, а чередовать примеры из каждого датасета, так модель улавливает больше полезного сигнала<br><br>
- *Постепенное размораживание*<br>начинать с последних слоёв, оставляя ранние нетронутыми. Полезно при адаптации к новым доменам; снижает падение метрик качества относительно полного дообучения

Многочисленные ислледования показывают, что современные модели избыточны по своим весам. то есть большую часть весов можно удалить и почти ничего не поменяется. На этой идее базируется целое направление - Pruning нейросестей. Его идея в том, чтобы после обучения эмпирически выделять бесполезные параметры и убирать их из модели, делая ее меньше. Текущие подходы не позволяет узнать до начала обучения, какой будет полезен, какой нет. Поэтому прунинг не решает проблему полностью, да и для практического использования подход неудобен, так как требует спеиального железа, умеющего работать с разреженностью. Нужно поискать какой-то другой подход

Итого, дообучение модели - дорогое, но его точно можно сделать более дешевым. Вопрос - как?

---

## Эффективное дообучение (PEFT)

PEFT (Parameter-Efficient Fine-Tuning) — это зонтичный термин для семейства подходов, объединённых общей идеей: заморозить большую часть весов предобученной модели и обучать лишь маленькую долю параметров (часто менее 1%). Логика проста: предобученная модель уже содержит почти всё нужное. Чтобы адаптировать её под задачу, не нужно перестраивать всю сеть — достаточно небольшого «довеска», который сдвигает поведение в нужную сторону.

Преимущества PEFT-подходов:
- обучаем мало параметров => нужно хранить мало градиентов и состояний оптимизатора => резко снижается потребление памяти при обучении
- дообученный «довесок» весит мегабайты вместо гигабайт => его легко хранить и подключать при необходимости к основной модели (аналог "картриджа" к приставке)
- базовые веса не трогаются => меньше риск переобучения на маленькой выборке и меньше catastrophic forgetting => модель работает точнее

Работа 2024 года с обзором PEFT методов<br>[Scaling Down to Scale Up: A Guide to Parameter-Efficient Fine-Tuning](https://arxiv.org/pdf/2303.15647)

### Таксономия PEFT-методов

Все PEFT-методы можно разложить по нескольким классам в зависимости от того, *что именно* в модели становится обучаемым. «добавляем новые параметры или выбираем существующие?», «если добавляем — модулем внутри сети или вектором на входе?», «меняем веса напрямую или через компактную репараметризацию?»

<img src="img/finetuning_peft.png" width=600>

Можно выделить следующие группы подходов:
- Selective (селективные): большую часть весов модели замораживаем, дообучаем только небольшой процент
- Additive (аддитивные): добавляем в модель новые параметры и обучаем только их
- Adapters (адаптеры): добавляем в модель адаптеры - небольшие двухслойные MLP сети, которые кодируют апдейт
- Soft prompts (мягкие промпты): присоединяем в модель фиксированное кол-во эмбедингов, отвечающих за режим работы модели
- Reparametrization-based (на основе репараметризации): добавляем в модель мультипликативное приближение
- Combinations (комбинации)

## Селективные методы
__Идея:__ давайте заморозим большую часть весов, а дообучать будем только некоторое небольшое подмножество  параметров модели, которые считаем наиболее важными. Это вероятно, самый простой подход

К плюсам можно отнести простоту подхода и то, что исходная архитектура никак не меняется: это значит что на этапе инференса нет дополниетнльых слоёв, которые нужно вычислять. Из минусов: выбор отдельных весов для зануления часто бывает неудобно эффективно реализовать на железе

Популярные методы:
- **BitFit** [(Zaken et al, 2022)](https://arxiv.org/abs/2106.10199)<br>Обучаем только bias-параметры<br><br>
- **LN Tuning** [(Zhao et al, 2020)](https://arxiv.org/abs/2312.11420)<br>Обучаем только параметры слоев нормализации<br><br>
- **Attention Tuning**<br>Обучаем только веса слоёв внимания<br><br>
- **Diff-Pruning** [(Guo et al, 2020)](https://arxiv.org/abs/2012.07463)<br>Мы хотим минимизировать ошибку на конечной задаче с занулением как можно большего числа параметров в векторе апдейта $\Delta W$. Зануление представим как умножение на бинраную маску $\Delta W^{pruned} = \Delta W \odot z$. Кол-во ненулевых элементов измеряется метрикой $L_0$, которая не дифференцируема => авторы предложили заменить бинарную маску (зануляем параметр / не зануляем) некоторым непрерывным выражением, идею которое предложили в [работе](https://arxiv.org/abs/1712.01312) 2018 года - там берут сигмоиду от случайного шума, немного растягивают её и применяем порог. И  оказывается, что матожидание единицы $P(z=1)$ очень просто выражается через единственный location-параметр $\alpha$ => все выражение легко оптимизировать<br>После обучения применяем порог и выбираем ненулевые параметры - это и будет вектор. Почему не $L_1$: хз<br><br>
- **Fish-Mask** [(Sung et al, 2021)](arxiv.org/abs/2111.09839)<br>FISH = Fisher-Induced Sparse uncHanging Mask. Отбираем важнейшие параметры по Fisher information, обучение только их<br><br>
- **LT-SFT** [(Ansell et al, 2023)](arxiv.org/abs/2110.07560)<br>LT-SFT = Lottery Ticket Supervised Fine-Tuning. На первой итерации дообучения смотрим, какие параметры изменились больше всего: $\arg \max \theta_1 - \theta_0$. На второй - повторяем цикл обучения, но уже только на них, остальные зануляем. Получившееся множество параметров и есть вектор изменения $\Delta W$<br><Br>Авторы повторяют принцип [Lottery Ticket Hypothesis](https://en.wikipedia.org/wiki/Lottery_ticket_hypothesis) известный в теории прунинга нейросестей и сформулированный в 2018 году, который говорит, что всегда есть небольшое подмножество пармаметров сети, обучение на котором дает почти исходное качество (говоря проще, нейросети в текущем их понимании всегда в каой=то степени избыточны). Этот принцип дал название подходу<br><br>
- **FAR** [(Vucetiuc et al, 2022)](arxiv.org/abs/2205.01541)<br>Метод Freeze and Reconfigure. Здесь работаем только с FFN модулями. Сначала прогоняем несколько итераций дообучения и выбираем top-k <u>нейронов</u>, которые будем занулять (топ определяем по изменчивости относительно начального веса). Нейроны = строки матрицы весов W. Кроме того, чтобы обеспечить sequential access при backpropagation оставшихся весов, перегруппируем нейроны по типу learner / non-learner<br><img src="img/finetuning_far.png" width=500>

---

## Аддитивные методы
Это пожалуй, самое широкое семейство. Идея в том, что на этапе дообучения все веса модели замораживаются, а в саму модель добавляются дополниетльные обучаемые параметры, которых не было в исходной архитектуре, чья роль "направлять" поведение модели в нужную сторону. Плата за гибкость - небольшая дополнительная задержка на инференсе, поскольку нужно обсчитывать больше параметров по сравнению с оригинальной моделью

- **Ladder-Side Tuning** [(Sung et al, 2022)](https://arxiv.org/abs/2206.06522)<br>отдельная маленькая «боковая» сеть обучается параллельно замороженной модели, получая её скрытые состояния; градиенты не идут через основную сеть → сильная экономия памяти<br><img src="img/ladder.png" width=150><br><br>
- **AttentionFusion** [(Cao et al, 2022)](https://www.amazon.science/publications/attention-fusion-a-light-yet-efficient-late-fusion-mechanism-for-task-adaptation-in-nlu)<br>Давайте брать все промежуточные выходы модели (не только выход последнего слоя), и комбинировать эти выходы в Attention стиле, как взвешенную сумму. Затем эта сумма прогоняется через небольшой MLP под специфическую задачу, под которую мы дообучаемся<br><img src="img/attention_fusion.png" width=300><br>Сравнение с некоторыми другими методами<br><img src="img/attention_fusion_1.png" width=600><br><br>
- **LeTS**  [(Fu, 2021)](https://proceedings.mlr.press/v139/fu21a.html)<br>Модель Learn To Share. Здесь также комбинируем выход со всех слоев, но уже не в Attention стиле, а чуть посложнее. Идут две параллельне ветки - "замороженная" предобученная и текущая, на каждом слое модель через бинарный селектор сама выбирает: а) какая из двух веток пойдет дальше б) какую ветку пустить на выход. Выход немного специфический: берем с каждого слоя эмбединг [CLS] токена, прогноняем через линейную трансофрмацию Linear и последовательно прокручиваем все эмбединги через Bi-LSTM начиная с первого<br><img src="img/lts1.png" width=400><br><br>Кроме того, на обучаемую ветку накладываем спрасификацию методом DeltaPruning, чтоба она разреженной была. Выбор селектора реализуется через подход хорошо известный в домене NAS (neural architecture search), и называемый DARTS. Идея в том, что мы сначала собираем модель со всеми опциями (так называемая "супермодель") и добавляем дискретный селектор между опциями. И чтобы не искать оптимум полным перебором, каждый дискретный переключатель заменяется неким непрерывным дифференцируемым приближением (конкретно в методе LeTS это взвешивание по Gumbalt: сигналы всех веток объединяются как взвешенная сумма). И теперь это приближение можно использовать при обучении сети через backpropagation - гиперпараметр стал параметром<br><br>Принимая во внимание сложность предлагаемой архтиектуры, очевидно, что модель в таком виде никто не использует

## Адаптеры
__Идея:__ давайте вставлять компактные обучаемые модули внутрь блоков трансформера. Они будут кодировать выученную "поправку" к предобученной модели. Базовые веса замораживаются, обучаются только эти вставки

- **Adapters** [(Houlsby, 2019)](https://arxiv.org/abs/1902.00751)<br>Одна из первых работ, описавшая архитектуру классического двухслойного адаптера. Адаптер здесь это двухслойная MLP сеть, собранная по принципу "бутылочного горлышка" (размерность посередине меньше входной и выходой):<br>`выход = вход + W_up · нелинейность( W_down · вход )`<br>В работе Houlsby адаптер вставляется в каждый слой сети в двух местах: на выходе Attention-модуля (но до LayerNorm) и на выходе FFN модуля (но до LayerNorm)<br>Обучается всего 3% параметров и при этом дает только -0.4% по качеству (сравнивали BERT модель на GLUE)<br><img src="img/adapters_houlsby.png" width=600><br><br>
- **Parallel Adapters** [(He et al, 2021)](https://arxiv.org/abs/2110.04366) <br>Используем такой же адаптер, но подключаем его параллельно, а не последовательно <br><img src="img/adapters_parallel.png" width=600><br><br>
- **AdapterFusion** [(Pfeiffer et al, 2021)](https://arxiv.org/pdf/2005.00247)<br>Авторы предложили механизм эффективного комбинирования нескольких адаптеров, и заодно показали, что для достижения того же уровня достаточно одного адаптера, размещенного после FFN модуля. Кроме того, LayerNorm слой тоже заморозили<br><img src="img/adapters_pfeiffer.png" width=200><br><br>
- **AdaMix** [(Wang et al, 2021)](https://arxiv.org/abs/2205.12410)<br>В этой архитектуре адаптер "размножается" на несколько параллельных версий. При обуении сигнал посылается случайно в одну из них. На инференс выход всех адаптеров усреднияется смесь нескольких адаптеров в духе mixture-of-experts; на инференсе усредняются<br><img src="img/adamix1.png" width=600><br><br>
- **Sparse Adapter** [(He et al., 2022)](https://arxiv.org/abs/2210.04284)<br>Вставляем в сеть классический адаптер, но перед обучением предварительно прогноняем его на нескольких батчах и зануляем бесполезные нейроны каким-то из методов прунинга (из того что авторы пробовали лучше всего сработал [SNIP](https://arxiv.org/abs/1810.02340)), чтобы сделать разреженным<br><img src="img/adapters_sparse.png" width=500><br><br>Зачем нужно разреживание, если адаптер и так изначально небольшой:<br>а) во-первых, регуляризация. Актуально, когда мало данных для дообучения<br>б) Во-вторых, конфигурация, которую авторы назвали <i>Large-Sparse</i>: выгоднее иметь большую разреженную модель, чем мальенкую плотную при равном кол-ве параметров, делаем адапетр пошире и применяем SNIP спарсификацию<br><br>
- **Compacter / PHM Adapter** [(Karimi, 2021)]( https://arxiv.org/abs/2106.04647) <br>Здесь все матрицы весов $W_{down}$ и $W_{up}$ представляем как кронекровское произведение $A \bigotimes B$. Аналогия - как будто назрезаем римскую пиццу. $A$ - малая матрица, $B$ - матрица побольше. Экономия по параметрам уже есть. Кроме того, для устойчивости представим как сумму нескольких таких произведений $\sum A_i \bigotimes B_j$. Далее все матрицы $A_i$ делаем shared в рамках всей сети. Но гулять так гулять, есть третья оптимизация: все матрицы B представляем как rank-1 промзведение двух векторов $B=p\times q$<br><img src="img/adapters_compacter1.png" width=500><br>Если посчитать: кронекер вместо $64 × 768 = 49152$ делает $(4 × 4) \bigotimes (16 × 192) = 3088$ параметров. Предположим, берем по 4 пары: 12355 параметров. Наконец, $B$ заменяем произведением $16 + 4 x (10+192) = 832$ параметра вместо 49355. Сжатие 59x<br><br>
- **(IA)³** [(Lui et al, 2022)](https://arxiv.org/abs/2205.05638)<br>Модель называется "Infused Adapter by Inhibiting and Amplifying Inner Activations" = встраиваемый адаптер, усиливающий/ослабляющий активации слоев. По сути является просто Scaler-ом, который добавляется в трех местах: на выходе K проектора, на выходе V проектора а также на выходе FFN. <br><img src="img/adapters_ia1.png" width=300><br>На обучении коээффициент масшиабирования подбирается, а после окончания обучения он вмердживается в матрицу весов (так как композиция линейных слоев = линейный слой)<br><img src="img/adapters_ia2.png" width=300>

## Soft prompts методы
__Идея:__ давайте новое "знание" не прибавлять к весам модели, как это делают адаптеры, а *конкатенировать* его с весами модели. И находясь уже в векторных описаниях это знание будет замешиваться с входным сигналом через механизм внимания (Self-Attention), как будто это обычные параметры. Конкатенировать будем со входом, но не с самим текстовым промптом, а с первым слоем - таблицей сырых эмбедингов

Визуально процесс похож на добавление в промпт новых воодных, поэтому назвали prompting. А поскольку конкатенируются не сырые токены, а их эмбединги, то "soft prompting". В литературе присоединяемые эмбединги называют иногда псевдотокенами или виртуальными токенами

Несколько популярных подходов:
- **Prompt-tuning**  [(Lester et al, 2021)](https://arxiv.org/abs/2104.08691)<Br>Простейший вариант. Заводится обучаемая матрица `P` размером `[k × d]` — это `k` «виртуальных токенов» (обычно 10–100) размерности модели `d`. На прямом проходе они *приписываются спереди* к эмбеддингам реального входа, и объединённая последовательность длины `k + n` идёт через трансформер как обычно; в attention-слое реальные токены «видят» префикс (как будто обычные токены) и подстраиваются под него. Обучается только `P`, добавление происходит один раз — на самом входе<br><img src="img/prompt.png" width=150><br><br>
- **Prefix-tuning**  [(Li et al, 2021)](https://arxiv.org/abs/2101.00190) <BR>То же самое, только обучаемые векторы добавляются *на каждом слое*, причём прямо в механизм внимания: к реальным key и value приклеиваются обучаемые префиксные K/V (свой набор на слой)<br><img src="img/prefix.png" width=200><br><br>
- **P-tuning**  [(Liu ert al, 2021)](https://arxiv.org/abs/2103.10385)<Br>Во время обучения модели эмбединги псевдотокенов перед конкатенацией с эмбедингами реальных токенов прогнояются через небольшую MLP сеть (таким образом взаимодейству/ют друг с другом и становятся контекстно-зависимыми). После обучения в итоговый словарь вставляют уже "провернутую" версию эмбедингов вместо оригинальных. Эта манипуляция делается для стабилизации обучения, получается лучший performance на конечный задачах (хз почему, но вероятно, благодаря MLP подбирается более оптимальная "геометрия" сигнала)<br><img src="img/ptuning.png" width=450><Br><br>
- __WARP__ [(Hambardzumyan et al. 2021)](https://arxiv.org/abs/2101.00121)<br>то же самое, что Prompt Tuning, но добавляется небольшое кол-во токенов (1-5) + добавляется еще HEAD. <br><br>
- __SPoT__ [(Vu et al, 2022)](https://arxiv.org/pdf/2110.07904)<br>перенос выученных soft-prompt'ов с задачи на задачу как инициализация

## Репараметризация
__Идея:__ мы не добавляем новые слои в сеть (как adapters) и не приписываем токены ко входу (как soft prompts), а перепараметризуем поправку к уже существующим весовым матрицам через произведение меньших матриц. Поправка `ΔW` к исходной матрице `W` представляется в компактной факторизованной форме, что резко снижает число обучаемых параметров. Базовые веса при этом заморожены

- **Intrinsic-SAID** [(Aghajanyan et al, 2021)](https://arxiv.org/abs/2012.13255)<br>Модель SAID = Structure-Aware Intrinsic Dimension. В работе «Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning»  авторы провели большое исследование, в котором показали, что у предобученных моделей очень низкая «внутренняя размерность» (intrinsic dimension) - то есть должна существовать низкоразмерная репараметризация, столь же эффективная для дообучения, как и полное пространство параметров<br><br>
Апдейт $\Delta W$ моделируется как "upscale" проекция из компактного вектора $θ^{d}$. Проекция $P: \mathbb{R}^d \rightarrow \mathbb{R}^D$ - случайная проекция, генерируемая один раз, реализованная через FastFood преобразование (математический трюк для быстрой генерации случайных матриц). В базовом варианте (DID: direct ID) одна проекция шарится между всеми слоями. В Structure-Aware варианте она для каждого слоя своя<br><br>
Fastfood трансформация выглядит вот так: $F(w_r) = \frac{1}{\sigma \sqrt{n}}BH\Pi GS$, где<br>
B – diagonal matrix with random ±1 entries<br>
H – Walsh–Hadamard matrix<br>
Π – random permutation matrix<br>
G – diagonal Gaussian random variables<br>
S – scaling factors for variance control<br>
σ – normalization constant

- __LoRA__ [(Hu et al, 2021)](https://arxiv.org/abs/2106.09685)<br>
Метод LoRA = Low-Rank Adaptation предложен в 2021 году авторами из Microsoft и стал де-факто стандартом дообучения LLM<br><br>Идея: апдейт $\Delta W$ низкоранговый => его можно апроксимировать произведением двух "узких" матриц A и B и обучать только их:$$h = W_{\theta}x + \Delta Wx = W_{\theta}x + (B \cdot A)x$$
где матрица $A \in \mathbb{R}^{(r×k)}$ - «up-projection», сжимает вход в ранг `r`, матрица $B \in \mathbb{R}^{(d×r)}$ — «up-projection», разворачивает обратно<br><img src="img/lora1.jpg" width=350><br><br>
Есть очевидное сходство с адаптерами, а точнее с паралельным его вариантом (Parallel adapter). Единственное отличие: LoRA это линейное преобразование, в то время как PA имеет актвиацию между слоями<br><img src="img/lora_adapter.png" width=350><Br>
Кроме того поправка иногда масштабируется коэффициентом $α/r$: $$h = W_{\theta}x + (α/r) \cdot B \cdot A \cdot x$$
Исследование авторов показало, что LoRA разложение выгоднее применять к Attention матрицам $W_q$ и $W_v$<Br><br>
Матрица $A$ инициализируется Гауссовым распредедением $A \propto N(0, \sigma^2)$, а матрица $B$ — нулями. То есть `BA = 0`, и модель начинает с нуля накапливать изменение<br><br>
После обучения можно слить с матрицами, либо хранить `{A, B}` отдельно как подключаемый адаптер<br><br>
Для GPT-3-175B LoRA снижает число обучаемых параметров в 10000 раз; потребление GPU-памяти в 3 раза; при том же качестве обучения. Ближайший конкурент - Houlsby адаптеры. На некоторых задачах качество даже выше fine-tuning.
<img src="img/lora2.png" width=550>

- __KronA__ [(Edalati et al, 2022))](https://arxiv.org/pdf/2212.10650)<br>
Метод KronA = Kronecker Adapter. Авторы из Huawei предложили модификацию принципа LoRA, где вместо матричного произведения используется произведение Кронекера.<br><br>Произведение Кронекера<br><img src="img/krona.png" width=250><br><br>
Проблема с LoRA в том, что ранг поправки жёстко ограничен сверху значением $r$. У произведения Кронекера есть хорошее свойство, что оно сохраняет ранг перемножаемых матриц: $rank(A \bigotimes B) = rank(A) \cdot rank(B)$<br><br>
Линейный слой с KronA: $h = W_{\theta}x + s·(A_k \bigotimes B_k) \cdot x$, где $s$ - масштабирующий коэффициент как в LoRA<br><br>
Адаптируется как Attention, так и FFN. Ниже вариант для Attention (слева - LoRA, справа - KronA)<br><img src="img/krona_lora1.png" width=350>
Вариант для FFN (слева - LoRA, справа - KronA). Здесь добавляется residual connection<br><img src="img/krona_lora2.png" width=350><br><br>
После обучения сливаем $\Delta W$ с $W_{\theta}$, либо если `KronAᴮ_res`, то оставить как параллельную ветвь с residual connection

- **QLoRA** (2023) <br>важное развитие. Идея: чтобы экономить память ещё сильнее, базовую модель **квантуют** — хранят её веса в очень низкой точности (4 бита вместо 16). Базовая модель при этом заморожена, а LoRA-адаптеры обучаются поверх неё в более высокой точности.
<br>Результат: дообучение очень крупных моделей становится возможным на одной потребительской видеокарте. QLoRA фактически демократизировала fine-tuning — то, что раньше требовало кластера, стало доступно энтузиастам<br><br>

---

Количество обучаемых параметров падает с `d²` до `2 · d · r`. При `d = 4096` и `r = 8` это уменьшение примерно в 256 раз.
<br>Переключаемость - можно обучить много разных LoRA-«адаптеров» под разные задачи (каждый весит мегабайты) и подгружать их к одной и той же базовой модели по необходимости. Это снимает проблему хранения десятков полных копий.

## Гибридные подходы
Не самостоятельный принцип, а методы на пересечении семейств. Два типичных сценария: ручное объединение механизмов в единый фреймворк (например, prefix-tuning + адаптеры) и обучаемое смешивание, когда метод сам через gating подбирает, какие механизмы и в какой пропорции включить. Отдельная ветвь — систематический поиск по «дизайн-пространству» PEFT.

- **MAM Adapter** [He et al, 2022](https://arxiv.org/abs/2110.04366)<br>разные части трансформера лучше адаптируются разными механизмами — внимание префиксами, FFN адаптером большой ёмкости
- **UniPELT** [(Mao et al, 2022)](https://arxiv.org/abs/2110.07577)<br>Метод Unipelt = A Unified Framework for Parameter-Efficient Language Model Tuning. Это попытка создать универсальное решение: в каждый блок трансформера встроены три PEFT-подмодуля сразу: LoRA + prefix + adapters, и над каждым стоит обучаемый «вентиль» (gate), определяющий его вклад<br><img src="img/unipelt.png" width=300><br><br>
- **IPT** [(Qin, 2022)](https://arxiv.org/abs/2110.07867)<br>Модель IPT=Intrinsic Prompt Tuning. Берем 100 задач и обучаем soft prompts для каждой. Учим на 100 точках автоэнкодер. Замораживае м получившийся декодер. сжимаем эти софт промпты в вектор
- **S4** [Chen, 2023](https://arxiv.org/abs/2301.01821)<br>результат систематического поиска по «дизайн-пространству» PEFT
- **Sparse LoRA**<br>LoRA с разреженностью (selective + reparametrization)

---

## Данные

В зависимости от того, чего мы хотим добиться:
- Continued pre-training (доменная адаптация)<br>Просто большой объём «сырого» текста из целевого домена (например, корпус юридических документов). Цель — пропитать модель доменным языком и фактами. Разметка не нужна<br><br>
- Supervised fine-tuning, SFT / instruction tuning<br>Пары «инструкция (вход) → желаемый ответ (выход)». Это основной формат для обучения модели следовать указаниям. Именно здесь формируется поведение «ассистента»<br><br>
- Данные предпочтений (preference data)<br>Тройки вида «запрос + лучший ответ + худший ответ». Используются на следующей стадии (выравнивание), которая выходит за рамки этой темы.

Для SFT часто также добавляют системный промпт и структуру диалога (роли «пользователь»/«ассистент»), чтобы модель училась работать в чат-формате

Отдельно стоит подчеркнуть значимость instruction tuning. Ранний fine-tuning настраивал модель под одну узкую задачу. Прорыв состоял в том, чтобы дообучить модель на смеси *множества разнообразных задач, сформулированных как инструкции на естественном языке*.

Результат оказался неожиданным: модель, обученная следовать инструкциям на наборе известных задач, начинает обобщать это умение и разумно реагировать на новые, не виденные ранее инструкции. Так «языковая модель, предсказывающая токены» превращается в «ассистента, выполняющего запросы». Так языковые модели стали привычным для нас чат-ботами.

Способы сбора данных для дообучения:

- Ручная разметка людьми<br>Эксперты или асессоры пишут эталонные ответы на запросы. Самый дорогой, но и самый качественный способ; хорош для задач, требующих экспертизы или аккуратности. Часто используется для сравнительно небольших, но «золотых» наборов.

- Сбор из существующих источников<br>Переиспользование уже имеющихся данных: логи поддержки, базы вопросов-ответов, документация, существующие датасеты. Дёшево, но требует очистки и приведения к нужному формату

- Синтетическая генерация (LLM-generated data)<br>Современный и очень популярный подход: использовать более сильную модель, чтобы сгенерировать обучающие данные для целевой. Типичный приём — дать мощной модели несколько примеров-«затравок» и попросить нагенерировать тысячи разнообразных пар «инструкция → ответ» (подход в духе Self-Instruct). Это дёшево и масштабируемо

- Важные оговорки по синтетике: нужно следить за разнообразием (модель склонна повторяться) и за накоплением ошибок (если генератор ошибается, его ошибки попадут в выборку). Часто синтетику комбинируют с человеческой фильтрацией.

- Distillation (дистилляция)<br>Частный случай синтетики: обучаем меньшую модель на ответах большей «учительской» модели, перенося её поведение. Многие открытые инструктивные модели обучены именно так.

Ключевой эмпирический вывод последних лет: для instruction tuning качество и разнообразие данных важнее их объёма. Несколько тысяч тщательно отобранных, разнообразных и чистых примеров часто дают лучший результат, чем сотни тысяч шумных. Это породило отдельное направление работы — отбор и фильтрацию данных (data curation): дедупликация, отсев низкокачественных и токсичных примеров, балансировка по типам задач, контроль длины и сложности.

При сборе выборки также важно следить за:
- покрытием - представлены ли все типы запросов, которые встретятся в проде;
- балансом - не доминирует ли один тип задач;
- утечками - не попали ли в обучение тестовые примеры (это завысит метрики);
- форматной согласованностью — единый стиль и структура ответов

---

## Резюме главы

Итого, что мы узнали. Full fine-tuning обновляет все веса — даёт максимум качества, но дорог по памяти и хранению, плохо масштабируется на большие модели. PEFT — общее семейство методов, экономящих ресурсы за счёт обучения лишь малой доли параметров. Таксономия по принципу «что обучается»: additive (добавляем новые параметры; внутри — adapters, модули в блоках, и soft prompts, виртуальные векторы на входе/в внимании), selective (обучаем подмножество существующих весов), reparametrization-based (компактная форма поправки к весам, сливаемая с базой) и комбинации. LoRA — доминирующий PEFT-метод: представляет поправку к весам как произведение двух низкоранговых матриц; обучает <1% параметров и не добавляет задержки на инференсе после слияния весов. QLoRA добавляет квантование базовой модели, делая дообучение крупных моделей доступным на скромном «железе».

Данные решают. Формат выборки зависит от цели (доменная адаптация / SFT / предпочтения). **Instruction tuning** превратил LLM в ассистентов. Данные собирают вручную, из готовых источников, синтетически или через дистилляцию — и для инструктивного дообучения качество и разнообразие важнее объёма